In [5]:

# ── 0. DEPENDENCIES ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import json, hashlib, os, sys
from datetime import datetime, timedelta
from typing import Any

import numpy as np
import pandas as pd


try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich.progress import track
    from rich import box
    from rich.text import Text
    RICH = True
except ImportError:
    RICH = False
    print("Tip: pip install rich  →  enables coloured console reports.")

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION  
# ─────────────────────────────────────────────────────────────────────────────

DEFAULT_CONFIG = {
   
    "source_name": "kafka-minio-pipeline",
    "source_version": "1.0",

    "completeness": {
        "required_columns": [],          
        "min_fill_rate": 0.95,           
        "column_fill_rates": {},         
    },


    "uniqueness": {
        "unique_columns": [],            
        "composite_key": [],             
        "max_duplicate_rate": 0.01,      
    },

  
    "range_checks": {
        # column_name: { "min": val, "max": val, "allowed": [list] }
        # Example:
        #   "age":    { "min": 0,   "max": 120 },
        #   "status": { "allowed": ["active", "inactive", "pending"] },
    },

    # ── statistical anomaly detection ──
    "anomaly": {
        "zscore_threshold": 3.0,         
        "iqr_multiplier": 1.5,           
        "method": "zscore",             
    },

    # ── scoring weights (must sum to 1.0) ──
    "weights": {
        "completeness": 0.35,
        "uniqueness": 0.30,
        "range": 0.20,
        "anomaly": 0.15,
    },

    # ── reporting ──
    "report": {
        "output_dir": ".",
        "save_csv_issues": True,
        "save_json_summary": True,
        "print_console": True,
        "fail_threshold": 0.80,          # overall score below this = FAIL
    },
}


# ─────────────────────────────────────────────────────────────────────────────
# 2. METRIC HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _zscore_outliers(series: pd.Series, threshold: float) -> pd.Series:
    """Return boolean mask of outliers via Z-score."""
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if numeric.std(ddof=0) == 0:
        return pd.Series(False, index=series.index)
    z = (pd.to_numeric(series, errors="coerce") - numeric.mean()) / numeric.std(ddof=0)
    return z.abs() > threshold


def _iqr_outliers(series: pd.Series, multiplier: float) -> pd.Series:
    """Return boolean mask of outliers via IQR fence."""
    numeric = pd.to_numeric(series, errors="coerce")
    q1, q3 = numeric.quantile(0.25), numeric.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - multiplier * iqr, q3 + multiplier * iqr
    return (numeric < lo) | (numeric > hi)


# ─────────────────────────────────────────────────────────────────────────────
# 3. CHECKER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

class CompletenessChecker:
    """Check null / missing rates per column."""

    def __init__(self, cfg: dict):
        self.cfg = cfg

    def check(self, df: pd.DataFrame) -> dict:
        results = {}
        global_min = self.cfg.get("min_fill_rate", 0.95)
        per_col = self.cfg.get("column_fill_rates", {})
        required = self.cfg.get("required_columns", list(df.columns))
        if not required:
            required = list(df.columns)

        issues = []
        col_scores = {}

        for col in required:
            if col not in df.columns:
                issues.append({"column": col, "issue": "column_missing", "detail": "Column not found in data"})
                col_scores[col] = 0.0
                continue

            null_count = int(df[col].isna().sum())
            total = len(df)
            fill_rate = 1.0 - (null_count / total) if total else 1.0
            threshold = per_col.get(col, global_min)
            passed = fill_rate >= threshold
            col_scores[col] = fill_rate

            if not passed:
                issues.append({
                    "column": col,
                    "issue": "low_fill_rate",
                    "fill_rate": round(fill_rate, 4),
                    "threshold": threshold,
                    "null_count": null_count,
                    "total_rows": total,
                    "detail": f"Fill rate {fill_rate:.1%} < threshold {threshold:.1%}",
                })

        score = float(np.mean(list(col_scores.values()))) if col_scores else 1.0
        results = {
            "dimension": "completeness",
            "score": round(score, 4),
            "column_scores": col_scores,
            "issues": issues,
            "summary": f"{len(issues)} column(s) below fill-rate threshold",
        }
        return results


class UniquenessChecker:
    """Check for duplicate rows and unique-column violations."""

    def __init__(self, cfg: dict):
        self.cfg = cfg

    def check(self, df: pd.DataFrame) -> dict:
        max_dup_rate = self.cfg.get("max_duplicate_rate", 0.01)
        unique_cols = self.cfg.get("unique_columns", [])
        composite = self.cfg.get("composite_key", [])
        issues = []
        subscores = []

        # ── full-row duplicates ──
        dup_mask = df.duplicated(keep=False)
        dup_count = int(dup_mask.sum())
        dup_rate = dup_count / len(df) if len(df) else 0
        row_score = max(0.0, 1.0 - dup_rate / max(max_dup_rate, 1e-9))
        subscores.append(row_score)

        if dup_rate > max_dup_rate:
            issues.append({
                "check": "full_row_duplicates",
                "duplicate_rows": dup_count,
                "duplicate_rate": round(dup_rate, 4),
                "threshold": max_dup_rate,
                "detail": f"{dup_count} fully duplicated rows ({dup_rate:.1%})",
            })

        # ── composite key duplicates ──
        if composite and all(c in df.columns for c in composite):
            comp_dup = df.duplicated(subset=composite, keep=False)
            comp_count = int(comp_dup.sum())
            comp_rate = comp_count / len(df) if len(df) else 0
            comp_score = max(0.0, 1.0 - comp_rate / max(max_dup_rate, 1e-9))
            subscores.append(comp_score)
            if comp_rate > max_dup_rate:
                issues.append({
                    "check": "composite_key_duplicates",
                    "key_columns": composite,
                    "duplicate_rows": comp_count,
                    "duplicate_rate": round(comp_rate, 4),
                    "detail": f"{comp_count} rows share a non-unique composite key",
                })

        # ── single-column unique checks ──
        for col in unique_cols:
            if col not in df.columns:
                continue
            col_dup = df.duplicated(subset=[col], keep=False)
            col_dup_count = int(col_dup.sum())
            col_dup_rate = col_dup_count / len(df) if len(df) else 0
            col_score = max(0.0, 1.0 - col_dup_rate / max(max_dup_rate, 1e-9))
            subscores.append(col_score)
            if col_dup_count > 0:
                issues.append({
                    "check": "unique_column_violation",
                    "column": col,
                    "duplicate_rows": col_dup_count,
                    "duplicate_rate": round(col_dup_rate, 4),
                    "detail": f"Column '{col}' contains {col_dup_count} non-unique values",
                })

        score = float(np.mean(subscores)) if subscores else 1.0
        return {
            "dimension": "uniqueness",
            "score": round(score, 4),
            "total_rows": len(df),
            "full_row_duplicates": dup_count,
            "issues": issues,
            "summary": f"{len(issues)} uniqueness violation(s) found",
        }


class RangeChecker:
    """Check numeric ranges and categorical domain constraints."""

    def __init__(self, cfg: dict):
        self.cfg = cfg  # {col: {"min":, "max":, "allowed":[]}}

    def check(self, df: pd.DataFrame) -> dict:
        issues = []
        col_scores = {}

        if not self.cfg:
            return {
                "dimension": "range",
                "score": 1.0,
                "issues": [],
                "summary": "No range rules configured",
            }

        for col, rules in self.cfg.items():
            if col not in df.columns:
                issues.append({"column": col, "issue": "column_missing"})
                col_scores[col] = 0.0
                continue

            violation_mask = pd.Series(False, index=df.index)

            if "min" in rules or "max" in rules:
                numeric = pd.to_numeric(df[col], errors="coerce")
                if "min" in rules:
                    violation_mask |= numeric < rules["min"]
                if "max" in rules:
                    violation_mask |= numeric > rules["max"]

            if "allowed" in rules:
                violation_mask |= ~df[col].isin(rules["allowed"]) & df[col].notna()

            viol_count = int(violation_mask.sum())
            viol_rate = viol_count / len(df) if len(df) else 0
            col_scores[col] = max(0.0, 1.0 - viol_rate)

            if viol_count > 0:
                issues.append({
                    "column": col,
                    "issue": "range_violation",
                    "violation_count": viol_count,
                    "violation_rate": round(viol_rate, 4),
                    "rules": rules,
                    "sample_bad_values": df.loc[violation_mask, col].head(5).tolist(),
                    "detail": f"{viol_count} values violate rule {rules}",
                })

        score = float(np.mean(list(col_scores.values()))) if col_scores else 1.0
        return {
            "dimension": "range",
            "score": round(score, 4),
            "column_scores": col_scores,
            "issues": issues,
            "summary": f"{len(issues)} column(s) with range/domain violations",
        }


class AnomalyChecker:
    """Detect statistical outliers in numeric columns."""

    def __init__(self, cfg: dict):
        self.cfg = cfg

    def check(self, df: pd.DataFrame) -> dict:
        method = self.cfg.get("method", "zscore")
        z_thresh = self.cfg.get("zscore_threshold", 3.0)
        iqr_mult = self.cfg.get("iqr_multiplier", 1.5)

        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        issues = []
        col_scores = {}
        all_outlier_counts = {}

        for col in numeric_cols:
            series = df[col]
            if series.dropna().nunique() < 4:
                col_scores[col] = 1.0
                continue

            if method in ("zscore", "both"):
                mask_z = _zscore_outliers(series, z_thresh)
            else:
                mask_z = pd.Series(False, index=df.index)

            if method in ("iqr", "both"):
                mask_iqr = _iqr_outliers(series, iqr_mult)
            else:
                mask_iqr = pd.Series(False, index=df.index)

            combined = mask_z | mask_iqr
            outlier_count = int(combined.sum())
            outlier_rate = outlier_count / len(df) if len(df) else 0
            col_scores[col] = max(0.0, 1.0 - outlier_rate)
            all_outlier_counts[col] = outlier_count

            if outlier_count > 0:
                numeric = pd.to_numeric(series, errors="coerce")
                issues.append({
                    "column": col,
                    "issue": "outliers_detected",
                    "outlier_count": outlier_count,
                    "outlier_rate": round(outlier_rate, 4),
                    "method": method,
                    "col_mean": round(float(numeric.mean()), 4),
                    "col_std": round(float(numeric.std()), 4),
                    "col_min": round(float(numeric.min()), 4),
                    "col_max": round(float(numeric.max()), 4),
                    "sample_outlier_values": df.loc[combined, col].head(5).tolist(),
                    "detail": f"{outlier_count} statistical outliers in '{col}'",
                })

        score = float(np.mean(list(col_scores.values()))) if col_scores else 1.0
        return {
            "dimension": "anomaly",
            "score": round(score, 4),
            "numeric_columns_checked": len(numeric_cols),
            "column_outlier_counts": all_outlier_counts,
            "issues": issues,
            "summary": f"{len(issues)} column(s) with anomalies detected",
        }


# ─────────────────────────────────────────────────────────────────────────────
# 4. REPORT GENERATOR
# ─────────────────────────────────────────────────────────────────────────────

class ReportGenerator:
    """Produce console + file reports from checker results."""

    GRADE_MAP = [
        (0.95, "A", "✅ EXCELLENT"),
        (0.85, "B", "✅ GOOD"),
        (0.75, "C", "⚠️  FAIR"),
        (0.60, "D", "⚠️  POOR"),
        (0.00, "F", "❌ CRITICAL"),
    ]

    def __init__(self, cfg: dict):
        self.cfg = cfg
        self.console = Console() if RICH else None

    def _grade(self, score: float) -> tuple:
        for threshold, letter, label in self.GRADE_MAP:
            if score >= threshold:
                return letter, label
        return "F", "❌ CRITICAL"

    def _score_color(self, score: float) -> str:
        if score >= 0.85:
            return "green"
        if score >= 0.70:
            return "yellow"
        return "red"

    # ── CONSOLE REPORT ──────────────────────────────────────────────────────

    def print_report(self, summary: dict):
        if not self.cfg.get("print_console", True):
            return

        ts = summary["generated_at"]
        overall = summary["overall_score"]
        letter, label = self._grade(overall)
        fail_thresh = self.cfg.get("fail_threshold", 0.80)
        status = "PASS" if overall >= fail_thresh else "FAIL"

        if RICH:
            self._rich_report(summary, overall, label, status)
        else:
            self._plain_report(summary, overall, label, status)

    def _plain_report(self, summary, overall, label, status):
        sep = "═" * 68
        print(f"\n{sep}")
        print(f"  DATA QUALITY REPORT  |  {summary['generated_at']}")
        print(f"  Source : {summary['source']}")
        print(f"  Rows   : {summary['row_count']:,}   Columns: {summary['col_count']}")
        print(sep)
        print(f"  OVERALL SCORE : {overall:.1%}  {label}  [{status}]")
        print(sep)

        for dim, res in summary["dimensions"].items():
            score = res["score"]
            letter, _ = self._grade(score)
            issues_n = len(res.get("issues", []))
            print(f"  {dim.upper():<14} {score:.1%}  (grade {letter})  |  {res['summary']}")

        print(sep)
        print("  TOP ISSUES:")
        all_issues = []
        for dim, res in summary["dimensions"].items():
            for iss in res.get("issues", []):
                all_issues.append((dim, iss))

        if not all_issues:
            print("  → No issues found. Data looks clean!")
        else:
            for i, (dim, iss) in enumerate(all_issues[:10], 1):
                detail = iss.get("detail", str(iss))
                col = iss.get("column", iss.get("check", ""))
                print(f"  {i:>2}. [{dim}] {col}: {detail}")

        print(sep)
        recommendations = summary.get("recommendations", [])
        if recommendations:
            print("  RECOMMENDATIONS:")
            for r in recommendations:
                print(f"   • {r}")
        print(sep + "\n")

    def _rich_report(self, summary, overall, label, status):
        c = self.console
        ts = summary["generated_at"]
        status_color = "green" if status == "PASS" else "red"

        c.print()
        c.rule("[bold cyan]DATA QUALITY REPORT[/bold cyan]")
        c.print(f"  [dim]Source:[/dim] [bold]{summary['source']}[/bold]  |  "
                f"[dim]Rows:[/dim] [bold]{summary['row_count']:,}[/bold]  |  "
                f"[dim]Cols:[/dim] [bold]{summary['col_count']}[/bold]  |  "
                f"[dim]{ts}[/dim]")
        c.print()

        score_color = self._score_color(overall)
        c.print(Panel(
            f"[bold {score_color}]{overall:.1%}  {label}[/bold {score_color}]\n"
            f"Status: [{status_color}]{status}[/{status_color}]",
            title="[bold]Overall Quality Score[/bold]",
            border_style=score_color,
        ))

        # dimension table
        table = Table(box=box.SIMPLE_HEAVY, show_lines=True)
        table.add_column("Dimension", style="bold cyan", min_width=14)
        table.add_column("Score", justify="right", min_width=8)
        table.add_column("Grade", justify="center", min_width=6)
        table.add_column("Issues", justify="right", min_width=7)
        table.add_column("Summary")

        for dim, res in summary["dimensions"].items():
            sc = res["score"]
            ltr, _ = self._grade(sc)
            clr = self._score_color(sc)
            n_iss = len(res.get("issues", []))
            table.add_row(
                dim.capitalize(),
                f"[{clr}]{sc:.1%}[/{clr}]",
                f"[{clr}]{ltr}[/{clr}]",
                str(n_iss),
                res.get("summary", ""),
            )
        c.print(table)

        # issues
        all_issues = []
        for dim, res in summary["dimensions"].items():
            for iss in res.get("issues", []):
                all_issues.append((dim, iss))

        if not all_issues:
            c.print("[green]✓ No data quality issues found.[/green]\n")
        else:
            iss_table = Table(title="Top Issues", box=box.SIMPLE, show_lines=True)
            iss_table.add_column("#", width=4)
            iss_table.add_column("Dimension", min_width=12)
            iss_table.add_column("Column / Check", min_width=16)
            iss_table.add_column("Detail")

            for i, (dim, iss) in enumerate(all_issues[:15], 1):
                col = iss.get("column", iss.get("check", "—"))
                detail = iss.get("detail", str(iss))
                iss_table.add_row(str(i), dim, str(col), detail)
            c.print(iss_table)

        # recommendations
        recs = summary.get("recommendations", [])
        if recs:
            c.print("[bold yellow]Recommendations[/bold yellow]")
            for r in recs:
                c.print(f"  • {r}")
        c.rule()
        c.print()

    # ── FILE EXPORT ─────────────────────────────────────────────────────────

    def save_json(self, summary: dict, path: str):
        with open(path, "w") as f:
            json.dump(summary, f, indent=2, default=str)
        print(f"[Report] JSON summary → {path}")

    def save_csv_issues(self, summary: dict, path: str):
        rows = []
        for dim, res in summary["dimensions"].items():
            for iss in res.get("issues", []):
                row = {"dimension": dim}
                row.update({k: v for k, v in iss.items() if not isinstance(v, (list, dict))})
                rows.append(row)
        if rows:
            pd.DataFrame(rows).to_csv(path, index=False)
            print(f"[Report] Issues CSV   → {path}")
        else:
            print("[Report] No issues to export to CSV.")


# ─────────────────────────────────────────────────────────────────────────────
# 5. MAIN ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────

class DataQualityMonitor:
    """
    Orchestrates all checkers and reporting.

    Usage
    -----
    monitor = DataQualityMonitor(config=DEFAULT_CONFIG)
    summary = monitor.run(df)
    """

    def __init__(self, config: dict = None):
        self.cfg = {**DEFAULT_CONFIG, **(config or {})}

    def _build_recommendations(self, dimensions: dict, overall: float) -> list:
        recs = []
        comp = dimensions.get("completeness", {})
        if comp.get("score", 1.0) < 0.90:
            recs.append("Run imputation or upstream data validation to reduce null rates.")
        uniq = dimensions.get("uniqueness", {})
        if uniq.get("full_row_duplicates", 0) > 0:
            recs.append("Deduplicate at the Kafka consumer level using a seen-key cache or exactly-once semantics.")
        rng = dimensions.get("range", {})
        if rng.get("score", 1.0) < 0.90:
            recs.append("Add schema-level constraints or Kafka Schema Registry validation to reject out-of-range values.")
        anom = dimensions.get("anomaly", {})
        if anom.get("score", 1.0) < 0.90:
            recs.append("Investigate upstream sensors / producers for anomalous readings; consider smoothing or flagging in MinIO metadata.")
        if overall < 0.75:
            recs.append("Overall quality is below acceptable threshold — halt downstream processing until issues are resolved.")
        if not recs:
            recs.append("Data quality looks healthy. Continue monitoring on a scheduled basis.")
        return recs

    def run(self, df: pd.DataFrame) -> dict:
        """Run all checks and return the full summary dict."""
        cfg = self.cfg
        weights = cfg["weights"]

        checkers = {
            "completeness": CompletenessChecker(cfg["completeness"]),
            "uniqueness":   UniquenessChecker(cfg["uniqueness"]),
            "range":        RangeChecker(cfg["range_checks"]),
            "anomaly":      AnomalyChecker(cfg["anomaly"]),
        }

        results = {}
        for name, checker in checkers.items():
            results[name] = checker.check(df)

        # weighted overall score
        overall = sum(results[k]["score"] * weights[k] for k in weights)

        summary = {
            "source": cfg["source_name"],
            "generated_at": datetime.now().isoformat(timespec="seconds"),
            "row_count": len(df),
            "col_count": len(df.columns),
            "columns": list(df.columns),
            "overall_score": round(overall, 4),
            "pass_threshold": cfg["report"]["fail_threshold"],
            "passed": overall >= cfg["report"]["fail_threshold"],
            "weights": weights,
            "dimensions": results,
            "recommendations": self._build_recommendations(results, overall),
        }

        reporter = ReportGenerator(cfg["report"])
        reporter.print_report(summary)

        # file exports
        out_dir = cfg["report"].get("output_dir", ".")
        ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")

        if cfg["report"].get("save_json_summary"):
            reporter.save_json(summary, os.path.join(out_dir, f"dq_summary_{ts_str}.json"))

        if cfg["report"].get("save_csv_issues"):
            reporter.save_csv_issues(summary, os.path.join(out_dir, f"dq_issues_{ts_str}.csv"))

        return summary


# ─────────────────────────────────────────────────────────────────────────────
# 6. KAFKA / MINIO DATA LOADERS  
# ─────────────────────────────────────────────────────────────────────────────

def load_from_kafka(
    bootstrap_servers: str,
    topic: str,
    max_records: int = 10_000,
    timeout_ms: int = 10_000,
) -> pd.DataFrame:
    """
    Read a batch of messages from a Kafka topic into a DataFrame.
    Requires: pip install confluent-kafka
    """
    try:
        from confluent_kafka import Consumer, KafkaException
    except ImportError:
        print("confluent-kafka not installed. Run: pip install confluent-kafka")
        return pd.DataFrame()

    consumer = Consumer({
        "bootstrap.servers": bootstrap_servers,
        "group.id": "dq-monitor-group",
        "auto.offset.reset": "earliest",
    })
    consumer.subscribe([topic])
    records = []
    try:
        while len(records) < max_records:
            msg = consumer.poll(timeout=timeout_ms / 1000)
            if msg is None:
                break
            if msg.error():
                raise KafkaException(msg.error())
            try:
                records.append(json.loads(msg.value().decode("utf-8")))
            except Exception:
                pass
    finally:
        consumer.close()

    return pd.DataFrame(records)


def load_from_minio(
    endpoint: str,
    bucket: str,
    object_name: str,
    access_key: str = "",
    secret_key: str = "",
) -> pd.DataFrame:
    """
    Load a Parquet or CSV object from MinIO into a DataFrame.
    Requires: pip install minio
    """
    import io
    try:
        from minio import Minio
    except ImportError:
        print("minio not installed. Run: pip install minio")
        return pd.DataFrame()

    client = Minio(endpoint, access_key=access_key, secret_key=secret_key, secure=False)
    response = client.get_object(bucket, object_name)
    data = response.read()

    if object_name.endswith(".parquet"):
        return pd.read_parquet(io.BytesIO(data))
    elif object_name.endswith(".csv"):
        return pd.read_csv(io.BytesIO(data))
    else:
        raise ValueError(f"Unsupported format for object: {object_name}")


# ─────────────────────────────────────────────────────────────────────────────
# 7. SYNTHETIC DEMO DATA  
# ─────────────────────────────────────────────────────────────────────────────

def generate_demo_data(n: int = 1_000, seed: int = 42) -> pd.DataFrame:
    """
    Generate a realistic-ish IoT event DataFrame with injected quality issues.
    Mirrors what you'd see from a Kafka → MinIO sensor pipeline.
    """
    rng = np.random.default_rng(seed)

    timestamps = pd.date_range("2024-01-01", periods=n, freq="1min")
    device_ids  = [f"device_{rng.integers(1, 51):03d}" for _ in range(n)]
    temperature = rng.normal(22, 3, n)
    humidity    = rng.uniform(30, 80, n)
    pressure    = rng.normal(1013, 5, n)
    voltage     = rng.normal(220, 10, n)
    status      = rng.choice(["active", "inactive", "pending", "unknown"], n, p=[0.7, 0.15, 0.12, 0.03])
    event_id    = [hashlib.md5(f"{ts}{did}".encode()).hexdigest()[:12]
                   for ts, did in zip(timestamps, device_ids)]

    df = pd.DataFrame({
        "event_id":    event_id,
        "timestamp":   timestamps,
        "device_id":   device_ids,
        "temperature": temperature,
        "humidity":    humidity,
        "pressure":    pressure,
        "voltage":     voltage,
        "status":      status,
    })

    # ── INJECT QUALITY ISSUES ──

    # 1. nulls (~7 % of temperature, 3 % of device_id)
    null_idx = rng.choice(n, size=int(n * 0.07), replace=False)
    df.loc[null_idx, "temperature"] = np.nan
    null_idx2 = rng.choice(n, size=int(n * 0.03), replace=False)
    df.loc[null_idx2, "device_id"] = np.nan

    # 2. duplicate rows (~2 %)
    dup_idx = rng.choice(n, size=int(n * 0.02), replace=False)
    df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

    # 3. temperature outliers
    outlier_idx = rng.choice(len(df), size=15, replace=False)
    df.loc[outlier_idx, "temperature"] = rng.choice([-50, 95, 120, -30], 15)

    # 4. pressure out of plausible range
    bad_pressure = rng.choice(len(df), size=10, replace=False)
    df.loc[bad_pressure, "pressure"] = rng.choice([200, 500, 50], 10)

    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# 8. ENTRY POINT 
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__" or True:   

    print("=" * 68)
    print("  DATA QUALITY MONITOR  —  Kafka → MinIO Pipeline")
    print("=" * 68)

  
    MY_CONFIG = {
        **DEFAULT_CONFIG,
        "source_name": "iot-sensor-stream-v1",

        "completeness": {
            "required_columns": ["event_id", "timestamp", "device_id",
                                  "temperature", "humidity", "pressure",
                                  "voltage", "status"],
            "min_fill_rate": 0.95,
            "column_fill_rates": {
                "event_id":   1.0,    
                "device_id":  1.0,
                "timestamp":  1.0,
            },
        },

        "uniqueness": {
            "unique_columns": ["event_id"],
            "composite_key":  ["device_id", "timestamp"],
            "max_duplicate_rate": 0.01,
        },

        "range_checks": {
            "temperature": {"min": -40, "max": 85},
            "humidity":    {"min": 0,   "max": 100},
            "pressure":    {"min": 900, "max": 1100},
            "voltage":     {"min": 190, "max": 260},
            "status":      {"allowed": ["active", "inactive", "pending"]},
        },

        "anomaly": {
            "method": "both",               
            "zscore_threshold": 3.0,
            "iqr_multiplier": 1.5,
        },

        "weights": {
            "completeness": 0.35,
            "uniqueness":   0.30,
            "range":        0.20,
            "anomaly":      0.15,
        },

        "report": {
            "output_dir":        ".",       
            "save_csv_issues":   True,
            "save_json_summary": True,
            "print_console":     True,
            "fail_threshold":    0.80,
        },
    }

    # ── Generate + assess demo data ─────────────────────────────────────────
    print("\n[Demo] Generating synthetic IoT sensor data …")
    demo_df = generate_demo_data(n=1_000)

    print(f"[Demo] Dataset shape : {demo_df.shape}")
    print(f"[Demo] Columns       : {list(demo_df.columns)}\n")

    # ── Run the monitor ─────────────────────────────────────────────────────
    monitor = DataQualityMonitor(config=MY_CONFIG)
    summary = monitor.run(demo_df)

    # ── Quick programmatic access to results ────────────────────────────────
    print("\n──────────────────────────────────────────────")
    print("  Programmatic result access (for automation):")
    print("──────────────────────────────────────────────")
    print(f"  overall_score : {summary['overall_score']:.1%}")
    print(f"  passed        : {summary['passed']}")
    for dim, res in summary["dimensions"].items():
        print(f"  {dim:<14} score={res['score']:.1%}  issues={len(res['issues'])}")

   

  DATA QUALITY MONITOR  —  Kafka → MinIO Pipeline

[Demo] Generating synthetic IoT sensor data …
[Demo] Dataset shape : (1020, 8)
[Demo] Columns       : ['event_id', 'timestamp', 'device_id', 'temperature', 'humidity', 'pressure', 'voltage', 'status']



─────────────────────────────────────────────── DATA QUALITY REPORT ───────────────────────────────────────────────

Source: iot-sensor-stream-v1  |  Rows: 1,020  |  Cols: 8  |  2026-04-28T16:50:49

╭───────────────────────────────────────────── Overall Quality Score ─────────────────────────────────────────────╮
│ 69.2%  ⚠️  POOR                                                                                                  │
│ Status: FAIL                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Dimension           Score   Grade     Issues   Summary                                   
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Completeness        98.8%     A            2   2 column(s) below fill-rate threshold     
                                                                                           
  Uniqueness           0.0%     F            3   3 uniqueness violation(s) found           
                                                                                           
  Range               99.1%     A            3   3 column(s) with range/domain violations  
                                                                                           
  Anomaly             98.9%     A            3   3 column(s) with anomalies detected

...

Recommendations

• Deduplicate at the Kafka consumer level using a seen-key cache or exactly-once semantics.

• Overall quality is below acceptable threshold — halt downstream processing until issues are resolved.

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

[Report] JSON summary → ./dq_summary_20260428_165049.json
[Report] Issues CSV   → ./dq_issues_20260428_165049.csv

──────────────────────────────────────────────
  Programmatic result access (for automation):
──────────────────────────────────────────────
  overall_score : 69.2%
  passed        : False
  completeness   score=98.8%  issues=2
  uniqueness     score=0.0%  issues=3
  range          score=99.1%  issues=3
  anomaly        score=98.9%  issues=3
